# Evaluating models on time series

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/x-datascience-datacamp/datacamp-master/blob/main/06_learning_with_time_series/01c-evaluating_on_time_series.ipynb)

```
Authors: Thomas Moreau
```

Licensed under [CC BY 4.0 Creative Commons](http://creativecommons.org/licenses/by/4.0/)

With time series, a careless evaluation can report excellent scores for a model
that will fail in production. There are two classic traps:

1. **Cross-validation that ignores time.** Consecutive samples are correlated,
   so a random split puts near-copies of the test points in the training set.
2. **Features that are not available at prediction time.** A feature computed
   from data that comes after the moment of the forecast leaks the future.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

URL_REPO = "https://github.com/x-datascience-datacamp/datacamp-master/raw/main/06_learning_with_time_series/"


def data_path(name):
    """Local file if the repo is cloned, its GitHub URL otherwise (Colab)."""
    path = Path("data") / name
    return path if path.exists() else f"{URL_REPO}data/{name}"

## 1. Cross-validation with autocorrelated data

We use the monthly airline passengers (see `01a`), in log scale. The only
feature is the time itself, counted in months. A nearest-neighbors regressor
predicts a month from the closest months in the training set.

In [ ]:
df = pd.read_csv(
    data_path("AirPassengers.csv"), parse_dates=["Month"], index_col="Month"
).asfreq("MS")
y = np.log(df["Passengers"])
X = np.arange(len(y)).reshape(-1, 1)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict, cross_val_score
from sklearn.neighbors import KNeighborsRegressor

model = KNeighborsRegressor(n_neighbors=2)
cv = KFold(n_splits=5, shuffle=True, random_state=0)

scores = cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error")
print(f"RMSE with shuffled K-fold: {-scores.mean():.3f}")

In [ ]:
y_pred = cross_val_predict(model, X, y, cv=cv)
ax = y.plot(label="true")
ax.plot(y.index, y_pred, label="cross-validated prediction")
ax.legend();

The predictions follow the series closely. Yet this model cannot forecast
anything: for any month after 1960, its two nearest neighbors are the last two
months of the training data.

The shuffled split violates the i.i.d. assumption behind cross-validation. Each
test month sits between training months that are strongly correlated with it
(see the autocorrelation plots in `01a`), so the score measures **interpolation**,
not **forecasting**.

<div class="alert alert-success">

**EXERCISE:**

- Evaluate the same model with `sklearn.model_selection.TimeSeriesSplit`, where each test fold comes after its training data.
- Plot the cross-validated predictions of each fold.

</div>

Solution is in: `solutions/eval-1_time-series-split.py`

The figure below compares the two splitting strategies. `TimeSeriesSplit` only
trains on the past of each test fold. Its `gap` parameter removes samples
between the training and the test sets, to mimic a delay between the last
available data and the forecast, and to reduce the correlation between the two
sets. `max_train_size` limits the training window (used in `02b`).

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

splitters = {
    "KFold(shuffle=True)": KFold(n_splits=5, shuffle=True, random_state=0),
    "TimeSeriesSplit()": TimeSeriesSplit(n_splits=5),
    "TimeSeriesSplit(gap=6)": TimeSeriesSplit(n_splits=5, gap=6),
}
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for ax, (name, splitter) in zip(axes, splitters.items()):
    for k, (train, test) in enumerate(splitter.split(X)):
        ax.scatter(train, [k] * len(train), c="C0", marker="_", lw=8)
        ax.scatter(test, [k] * len(test), c="C1", marker="_", lw=8)
    ax.set_title(name)
    ax.set_ylabel("fold")
axes[-1].set_xlabel("month index (blue: train, orange: test)")
plt.tight_layout()

## 2. Features that leak the future

We now forecast the hourly number of bikes on boulevard de Sébastopol (see `01a`
and `02c`). The use case: **every night at midnight, we forecast the 24 hours of
the next day**.

We predict the log of the counts, so the RMSE below is the RMSLE used in `02c`.

In [ ]:
bikes = pd.read_parquet(data_path("bike-counter-data.parquet"))
counts = (
    bikes.query("counter_name == 'Totem 73 boulevard de Sébastopol S-N'")
    .set_index("date")["bike_count"]
    .sort_index()
)
y = np.log1p(counts.asfreq("h").interpolate())

Besides calendar features, the past values of the series (lags) are natural
features: `y.shift(k)` is the value $k$ hours earlier.

In [ ]:
X = pd.DataFrame({
    "hour": y.index.hour,
    "weekday": y.index.day_of_week,
    "month": y.index.month,
}, index=y.index)
for lag in [1, 2, 24, 168]:
    X[f"lag_{lag}h"] = y.shift(lag)

# the first week has no lag_168h
X, y = X.iloc[168:], y.iloc[168:]
X.head()

We evaluate with a `TimeSeriesSplit` whose test folds are 4 weeks long.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor(random_state=0)
cv = TimeSeriesSplit(n_splits=5, test_size=4 * 7 * 24)


def rmsle(columns):
    scores = cross_val_score(
        model, X[columns], y, cv=cv, scoring="neg_root_mean_squared_error"
    )
    return -scores.mean()


calendar = ["hour", "weekday", "month"]
print(f"calendar only:    {rmsle(calendar):.3f}")
print(f"calendar + lags:  {rmsle(list(X.columns)):.3f}")

The lags help a lot. But the split is temporal, so is this score trustworthy?

<div class="alert alert-success">

**EXERCISE:**

- At midnight, which of these lags are known for each of the 24 hours to forecast? For instance, what is `lag_1h` when forecasting 18:00?
- Evaluate the model with the features that are available at prediction time only. How much of the gain was real?

</div>

Solution is in: `solutions/eval-2_forecast-horizon.py`

A lag is only usable if it is at least as long as the forecast horizon. The
temporal split did not catch this leak: cross-validation checks how the samples
are split, not what each feature contains.

The same leak takes many forms:

- a rolling or centered mean that includes the current or future values,
- a filling of missing values with `bfill` or interpolation over the test period,
- external covariates measured after the fact (the observed weather instead of
  the weather forecast available the night before),
- a scaler or target encoder fit on the full series before splitting.

**Checklist**: for each feature, ask *"would I know this value at the moment I
make the forecast?"*, and evaluate with a split that respects time, with a test
period that matches the horizon of the use case.